# AEM4L5 E06 — Optimizar pipeline lento

**Objetivo:** perfilar, detectar hotspot y refactorizar con evidencia.

**Tipo:** Para resolver.


## Flujo de diagnóstico

```mermaid
flowchart TD
    A["Pipeline lento"] --> B["Medir con cProfile"]
    B --> C["Ordenar por cumtime"]
    C --> D{"Hotspot"}
    D -->|"CPU-bound"| E["Optimizar código / regex / vectorizar"]
    D -->|"I/O-bound"| F["Async / batching / retries"]
    E --> G["Medir otra vez"]
    F --> G
```


In [ ]:
import cProfile
import io
import pstats
import re
import time

texts = ["Texto de prueba!!! con símbolos ### y cláusulas legales."] * 10000


def normalize_lento(texts):
    result = []
    for text in texts:
        clean = ""
        for char in text:
            if char.isalnum() or char.isspace():
                clean += char.lower()
        result.append(clean)
    return result


def profile_call(label, func):
    profiler = cProfile.Profile()
    start = time.perf_counter()
    profiler.enable()
    output = func(texts)
    profiler.disable()
    elapsed = time.perf_counter() - start
    stream = io.StringIO()
    pstats.Stats(profiler, stream=stream).sort_stats("cumtime").print_stats(8)
    print(f"\n{label}: {elapsed:.3f}s, items={len(output)}")
    print(stream.getvalue())
    return elapsed

# TODO 1: perfilar normalize_lento(texts)
# TODO 2: crear normalize_optimizado usando regex compilada
# TODO 3: comparar tiempos y explicar el hotspot
baseline = profile_call("baseline lento", normalize_lento)


## Criterios de solución

Una buena respuesta identifica el hotspot, cambia una sola cosa importante, vuelve a medir y explica por qué la mejora aplica a CPU-bound.
